# Demo 17 - Entity investigation timeline

**Pool:** Medium · **Visual:** multi-source swimlane timeline

**The question:** what did this device actually do, across everything we collect?

Investigation is inherently multi-table. Processes, network connections, logons and file
activity all live separately, and the pivot between them is what an analyst spends the day
doing. This notebook pulls all four for one entity and draws them as a single timeline with
a row per source.

KQL can union the tables. What it does not give you is the one annotated picture, or the
ability to keep pivoting in place as the investigation moves.

## 1. Connect to the data lake

`MicrosoftSentinelProvider` is the bridge between this notebook and your lake. The `spark`
session is handed to you by the Microsoft Sentinel kernel, so never create your own.

Nothing is read yet. This cell only opens the connection.

In [ ]:
from sentinel_lake.providers import MicrosoftSentinelProvider
from pyspark.sql import functions as F
data_provider = MicrosoftSentinelProvider(spark)

## 2. Parameters - which entity to investigate

Set `ENTITY` to the device you want the full picture of.

If that device does not exist in this tenant, the next cell automatically falls back to the
busiest device it can find, so the notebook always has something to show.

In [ ]:
# Parameters
WORKSPACE = "your-workspace-name"
ENTITY    = "zava-fin-01"        # device name to investigate
LOOKBACK_DAYS = 7

## 3. Pull everything that entity did, from four separate tables

An investigation is inherently multi-table. What ran? What did it talk to? Who logged in?
What files moved?

Each table becomes one "lane" with a common shape (time, source, detail), and the lanes are
unioned into a single timeline. Tables missing from this workspace, or missing a column we
need, are skipped with a message rather than failing the run.

Each lane is capped at 2,000 events so one busy server cannot flood the picture.

In [ ]:
import pandas as pd

def lane(tbl, keycol, detailcols, entity):
    try:
        t = data_provider.read_table(tbl, WORKSPACE)
        missing = [c for c in [keycol] + detailcols if c not in t.columns]
        if missing:
            print(f"(skip {tbl}: missing columns {missing})"); return None
        t = (t.filter(F.col("TimeGenerated") >= F.expr(f"current_timestamp() - INTERVAL {int(LOOKBACK_DAYS)} DAYS"))
               .filter(F.col(keycol) == entity))
        detail = F.concat_ws(" | ", *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in detailcols])
        return (t.select(F.col("TimeGenerated").alias("time"),
                         F.lit(tbl).alias("source"), detail.alias("detail"))
                 .limit(2000))
    except Exception as e:
        print(f"(skip {tbl}: {e})"); return None

LANES = [
    ("DeviceProcessEvents", "DeviceName", ["FileName","ProcessCommandLine"]),
    ("DeviceNetworkEvents", "DeviceName", ["RemoteIP","RemoteUrl"]),
    ("DeviceLogonEvents",   "DeviceName", ["AccountName","LogonType"]),
    ("DeviceFileEvents",    "DeviceName", ["ActionType","FileName"]),
]

def build_timeline(entity):
    frames = [f for f in (lane(t, k, d, entity) for t, k, d in LANES) if f is not None]
    if not frames:
        return pd.DataFrame(columns=["time","source","detail"])
    tl = frames[0]
    for f in frames[1:]:
        tl = tl.unionByName(f)
    return tl.toPandas()

pdf = build_timeline(ENTITY)

# If the sample device isn't in this tenant, investigate the busiest one instead so the
# demo always has a timeline to show.
if pdf.empty:
    print(f"No events for '{ENTITY}' - falling back to the busiest device.")
    busiest = (data_provider.read_table("DeviceProcessEvents", WORKSPACE)
               .filter(F.col("TimeGenerated") >= F.expr(f"current_timestamp() - INTERVAL {int(LOOKBACK_DAYS)} DAYS"))
               .filter(F.col("DeviceName").isNotNull())
               .groupBy("DeviceName").agg(F.count("*").alias("n"))
               .orderBy(F.desc("n")).limit(1).collect())
    if busiest:
        ENTITY = busiest[0]["DeviceName"]
        print("Investigating:", ENTITY)
        pdf = build_timeline(ENTITY)

if not pdf.empty:
    pdf["time"] = pd.to_datetime(pdf["time"])
print(f"events for {ENTITY}:", len(pdf))
pdf.sort_values("time").head(20)

## 4. Draw the swimlane timeline

One row per data source, one dot per event, plotted against time.

**What to look for:** vertical alignment. A process event, a network connection and a file
write all landing within the same few seconds is a causal chain, and the eye picks that out
of a swimlane far faster than out of a sorted table.

Gaps matter too. A device that goes quiet across every lane at the same moment either got
shut down or stopped reporting, and those are different problems.

In [ ]:
import matplotlib.pyplot as plt

if not pdf.empty:
    sources = sorted(pdf["source"].unique())
    ymap = {s:i for i,s in enumerate(sources)}
    colors = plt.cm.tab10.colors
    plt.figure(figsize=(14, 4.5))
    for s in sources:
        d = pdf[pdf["source"]==s]
        plt.scatter(d["time"], [ymap[s]]*len(d), s=30, alpha=.6,
                    color=colors[ymap[s] % 10], label=s)
    plt.yticks(range(len(sources)), sources)
    plt.title(f"Investigation timeline - {ENTITY} (last {LOOKBACK_DAYS} days)")
    plt.xlabel("time"); plt.legend(loc="upper left", fontsize=8); plt.tight_layout(); plt.show()
else:
    print(f"No events for {ENTITY} in window - check the device name.")

## Why this is a notebook hunt, not a KQL query

Investigation is inherently multi-table. A notebook unions processes, network, logons and files into one swimlane and keeps the pivot interactive. KQL can `union`, but the single annotated visual timeline - and the ability to keep drilling in-place - is the notebook advantage.